<a href="https://colab.research.google.com/github/TheraMind-Project-Team/Psychologist-Project-AI/blob/main/Hypired_text%26audioipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, r2_score

L0ad

In [ ]:

df = pd.read_csv('/content/drive/MyDrive/hybrid_cleaned_data.csv')



audio_scaler    = joblib.load('/content/drive/MyDrive/audio_scaler.pkl')
selector_25     = joblib.load('/content/drive/MyDrive/selector_251')
selector_4      = joblib.load('/content/drive/MyDrive/audio_selector_4.pkl')
audio_clf_model = joblib.load('/content/drive/MyDrive/model_clf_audio')
audio_hybrid_sc = joblib.load('/content/drive/MyDrive/audio_hybrid.pkl')
audio_reg_model = joblib.load('/content/drive/MyDrive/audio_reg_hybrid.pkl')
text_pipeline   = joblib.load('/content/drive/MyDrive/regressor_pipeline.pkl')

In [ ]:
audio_cols = [
    "mean_F2","mean_F3","mean_F4","mean_F1","mean_F5",
    "std_F2", "std_F3", "std_F4", "std_F1", "std_F5",
    "min_F3", "min_F4", "min_F1", "min_F5",
    "max_F2", "max_F3", "max_F4", "max_F1",
    "mean_C1","mean_C2","mean_C3","mean_C4","mean_C5",
    "mean_C6","mean_C7","mean_C8","mean_C9","mean_C10",
    "std_C1", "std_C2", "std_C3", "std_C4", "std_C5",
    "std_C6", "std_C7", "std_C8", "std_C9", "std_C10",
    "min_C5", "min_C8",
    "max_C1", "max_C3", "max_C4", "max_C5", "max_C6",
    "max_C7", "max_C8", "max_C9", "max_C10"
]

X_audio_raw  = df[audio_cols].values
X_text_raw   = df['Cleaned_Text'].fillna('').values
y_true_score = df['PHQ8_Score'].values


all_49_sc = audio_scaler.transform(X_audio_raw)

f_25 = selector_25.transform(all_49_sc)
f_4 = selector_4.transform(all_49_sc)

is_depressed   = audio_clf_model.predict(f_25).reshape(-1, 1)
prob_depressed = audio_clf_model.predict_proba(f_25)[:, 1].reshape(-1, 1)

hybrid_input = np.hstack((f_4, is_depressed, prob_depressed))
hybrid_input = audio_hybrid_sc.transform(hybrid_input)

v_scores = np.clip(audio_reg_model.predict(hybrid_input), 0, 24)

X_tfidf  = text_pipeline['tfidf'].transform(X_text_raw)
t_scores = np.clip(text_pipeline['reg'].predict(X_tfidf), 0, 24)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Avag

In [ ]:
phq8_estimates = (v_scores + t_scores) / 2

THRESHOLD = 9
y_pred_binary = (phq8_estimates >= THRESHOLD).astype(int)
y_true_binary = (y_true_score  >= THRESHOLD).astype(int)

print("\n" + "="*50)
print("Hybrid Model Test Results")
print("="*50)
print(f"MAE (Score): {mean_absolute_error(y_true_score, phq8_estimates):.2f} points")

print(f"R-squared (R2): {r2_score(y_true_score, phq8_estimates):.4f}")
print(f"Accuracy (Threshold >= {THRESHOLD}): {accuracy_score(y_true_binary, y_pred_binary)*100:.2f}%")
print("-"*50)
print(classification_report(y_true_binary, y_pred_binary,
                             target_names=['Normal (<9)', 'Depressed (>=9)']))


Hybrid Model Test Results
MAE (Score): 2.54 points
R-squared (R2): 0.7023
Accuracy (Threshold >= 9): 90.37%
--------------------------------------------------
                 precision    recall  f1-score   support

    Normal (<9)       0.91      0.94      0.93       120
Depressed (>=9)       0.89      0.84      0.86        67

       accuracy                           0.90       187
      macro avg       0.90      0.89      0.89       187
   weighted avg       0.90      0.90      0.90       187



# ***T***

In [ ]:
phq8_estimates = (0.62 * t_scores) + (0.38 * v_scores)

THRESHOLD = 9
y_pred_binary = (phq8_estimates >= THRESHOLD).astype(int)
y_true_binary = (y_true_score  >= THRESHOLD).astype(int)

print("\n" + "="*50)
print("Hybrid Model Test Results")
print("="*50)
print(f"MAE (Score): {mean_absolute_error(y_true_score, phq8_estimates):.2f} points")

print(f"R-squared (R2): {r2_score(y_true_score, phq8_estimates):.4f}")
print(f"Accuracy (Threshold >= {THRESHOLD}): {accuracy_score(y_true_binary, y_pred_binary)*100:.2f}%")
print("-"*50)
print(classification_report(y_true_binary, y_pred_binary,
                             target_names=['Normal (<9)', 'Depressed (>=9)']))


Hybrid Model Test Results
MAE (Score): 2.18 points
R-squared (R2): 0.7659
Accuracy (Threshold >= 9): 91.44%
--------------------------------------------------
                 precision    recall  f1-score   support

    Normal (<9)       0.91      0.96      0.93       120
Depressed (>=9)       0.92      0.84      0.88        67

       accuracy                           0.91       187
      macro avg       0.92      0.90      0.90       187
   weighted avg       0.91      0.91      0.91       187

